# 3. Integrate with Agent Framework (MCP tools, Skills, ...)

Microsoft Agent Framework is a library that helps building your agents, which provides unified programming that abstracts background clients - such as, Anthropic Claude, OpenAI, Microsoft Foundry, ...<br>
Agent Framework also provides client abstraction for Foundry Local.

By the benefit of this programming architecture, you can work with various local features - such as, local memory, client-side tool calling (including MCP tool calling), agent skills, ... - with a small number of lines of code through Agent Framework.

> Note : See [here (Agent Framework workshop)](https://github.com/tsmatsuz/agent-framework-workshop-with-foundry) for core features in Microsoft Agent Framework.

In this lesson, we use Foundry Local through Agent Framework and see the benefits to use Agent Framework.

> Note : The following ```FoundryLocalClient``` doesn't use web service, which is discussed in [Lesson2](./02_run_as_web_service.ipynb).<br>
> ```FoundryLocalClient``` directly calls Foundry Local SDK functions.

---

**Imporatnt Note** :<br>
Currently (on Agent Framework version 1.1.0), Foundry Local SDK 1.0 and above are not supported. Please use and install Foundry Local SDK 0.5.1 as follows.

```
winget install Microsoft.FoundryLocal
pip install foundry-local-sdk==0.5.1
```
---

## Local function calling

In the first example, we implement local function calling on Foundry Local through Agent Framework.

First of all, we initialize ```FoundryLocalClient``` in Agent Framework, which client works with Foundry Local SDK at the bottom.

> Note : ```qwen2.5-14b``` is a little large and it will take a long time to complete.

In [1]:
from agent_framework.foundry import FoundryLocalClient

client = FoundryLocalClient(model="qwen2.5-14b")

Next we define functions as local tools as follows.

In [2]:
from agent_framework import tool
from typing import Annotated
from pydantic import Field
from random import randint

@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return conditions[randint(0, 3)]

@tool(approval_mode="never_require")
def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],
) -> str:
    """Get the temperature for a given location."""
    return f"{randint(10, 30)} degrees"

Now let's run your agent as follows.

As you saw in [Lesson1](./01_get_started.ipynb), it was very tedious to process agentic loop - in which you should check responses, call local functions, build messages, and invoke requests again.<br>
Using Agent Framework, however, you can write in a much more intuitive way as follows.

In [3]:
from agent_framework import Agent

agent = Agent(
    name="BasicWeatherAgent",
    client=client,
    instructions=(
        "You are a helpful assistant with access to tools. "
        "Use them when needed to answer questions accurately."
    ),
    tools=[get_weather, get_temperature])

result = await agent.run("Tell me the weather in Seattle.")

print(result.messages[-1].text)

The weather in Seattle is cloudy.


## MCP tool calling

Agent Framework also provide client-side MCP tool callings.

MCP tool calling provided as APIs by cloud LLM vendors (such as, OpenAI, Anthropic, ...) are mostly processed on the server side. Foundry Local, however, does not have these rich server-side functionalities.

Instead, you can use client-side implementation in Agent Framework.<br>
The client-side MCP tool calling in Agent Framework is implemented as local function calling, and it then communicates with MCP servers using standard MCP protocol specification.

The following code uses [Microsoft Learn MCP server](https://github.com/mcp/microsoftdocs/mcp), which can retrive information about Microsoft official technical documentation.

In [4]:
from agent_framework import MCPStreamableHTTPTool

mslearn_mcp = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
    load_prompts=False,
    approval_mode="never_require",
)

agent = Agent(
    client=client,
    instructions=(
        "You are a helpful assistant with access to tools. "
        "Use them when needed to answer questions accurately."
    ),
    tools=[mslearn_mcp],
)

result = await agent.run("What is Microsoft Agent Framework?")
print(result.messages[-1].text)

Microsoft Agent Framework is a development platform provided by Microsoft that enables the creation of AI-driven applications. It consists of two main categories of capabilities:

1. Agents: These are individual AI entities that utilize Large Language Models (LLMs) to process inputs, interact with tools and hosted Model Client Protocol (MCP) servers, and generate responses. They support various providers like Microsoft Foundry, Anthropic, Azure OpenAI, OpenAI, Ollama, and more.

2. Workflows: These are graph-based workflows that connect agents and functions for multi-step tasks, offering type-safe routing, checkpointing, and human-in-the-loop support.

The framework also provides foundational building blocks, such as model clients (chat completions and responses), an agent session for state management, context providers for agent memory, middleware for intercepting agent actions, and MCP clients for tool integration. These components allow developers to build interactive, robust, and s

When you see the list of the processed messages, you can find that ```microsoft_docs_search``` tool is used to retrieve official information about "Microsoft Agent Framework".

In [5]:
import agent_framework

for i, msg in enumerate(result.messages):
    print(f"********** message {i} **********")
    for c in msg.contents:
        if c.type == "function_call":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function name : {c.name}")
            print(f"function arguments : {c.arguments}")
        elif c.type == "function_result":
            print(f"*** {c.type} ***")
            print(f"call id : {c.call_id}")
            print(f"function result : {c.result}")
            print(f"exceptions : {c.exception}")
        elif c.type == "text":
            print(f"*** {c.type} ***")
            print(f"text : {c.text}")
        else:
            print(f"*** Other types : {c.type}***")

********** message 0 **********
*** text ***
text :  Александа, за да отговоря на твоята въпроса за Microsoft Agent Framework, трябва първо да търся информация в официалната документация на Microsoft. Ще използвам функцията за търсене в документацията, за да намеря подходяща информация.
<tool_call>
{"name": "microsoft_docs_search", "arguments": {"query": "Microsoft Agent Framework"}}
</tool_call>
*** function_call ***
call id : cI00tPD4J
function name : microsoft_docs_search
function arguments : {
  "query": "Microsoft Agent Framework"
}
********** message 1 **********
*** function_result ***
call id : cI00tPD4J
function result : {"results":[{"title":"Microsoft Agent Framework (programming-language-csharp)","content":"# Microsoft Agent Framework (programming-language-csharp)\nAgent Framework offers two primary categories of capabilities:\n| - | Description| \n|  --- | ---  |\n| **[Agents](https://learn.microsoft.com/agent-framework/agents/)** | Individual agents that use LLMs to proces